In [ ]:
!pip install -q captum torchvision matplotlib pillow monai split-folders[full]

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
from PIL import Image
import timm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import ImageFolder
import torchvision.transforms as transforms
from torchvision.transforms import v2
from torchvision.datasets import ImageFolder
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import f1_score, roc_auc_score, confusion_matrix, classification_report, roc_curve
from sklearn.metrics import ConfusionMatrixDisplay
from monai.losses import FocalLoss
import splitfolders

from sklearn.metrics import f1_score, confusion_matrix, classification_report, roc_auc_score, roc_curve, ConfusionMatrixDisplay
import matplotlib.gridspec as gridspec

from captum.attr import IntegratedGradients, GuidedGradCam, Occlusion

# Set random seeds for reproducibility
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    
seed_everything(42)


# Global Constants
DATA_DIR = "/kaggle/input/datasets/neuralthon/spark-hackathon-2026-tb/TB/niaid-cxr"
OUTPUT_DIR = '/kaggle/working/output_folder'
IMG_SIZE = 380
BATCH_SIZE = 32
NUM_CLASSES = 2
EPOCHS = 5 #to be changed later
LEARNING_RATE = 1e-4


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using Device: {DEVICE}")

import warnings
warnings.filterwarnings("ignore")

print("Libraries imported succesfully!")

In [ ]:
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

# Data augmentation for training to improve model generalization
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),           # Randomly flip horizontally
    transforms.RandomRotation(15),                    # Slight rotations
    transforms.ColorJitter(brightness=0.2, contrast=0.2), # Brightness & Contrast
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.4818, 0.4818, 0.4818],
        std=[0.2223, 0.2223, 0.2223]
    )
])

# Standard transforms for validation and testing (No Augmentation)
val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
        transforms.Normalize(
        mean=[0.4818, 0.4818, 0.4818],
        std=[0.2223, 0.2223, 0.2223]
    )
])

print("Transforms Initialized")

In [ ]:
splitfolders.ratio(
    DATA_DIR, 
    output=OUTPUT_DIR,
    seed=42, ratio=(.8, 0.2),
    group_prefix=None, move=False
)

In [ ]:
train_dir = os.path.join(OUTPUT_DIR, 'train')
val_dir = os.path.join(OUTPUT_DIR, 'val')

train_dataset = ImageFolder(root=train_dir, transform=train_transform)
val_dataset   = ImageFolder(root=val_dir, transform=val_test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)

# Count class frequencies in training set
train_labels = [label for _, label in train_dataset]
num_normal = train_labels.count(0)
num_pneumonia = train_labels.count(1)
pos_weight = torch.tensor([num_normal / num_pneumonia]).to(DEVICE)
print(f"pos_weight = {pos_weight.item():.4f}")

val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
CLASS_NAMES = train_dataset.classes
print(f"DataLoaders built successfully, Classes: {CLASS_NAMES}")
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")
print(f'Total dataset size: {len(train_dataset) + len(val_dataset)}')
print(train_dataset.class_to_idx)

In [ ]:
image, label = next(iter(train_loader))
plt.imshow(image[1, 0, :, :], cmap="gray")

In [ ]:
class DenseNetClassifier(nn.Module):
    def __init__(self, num_classes=2, pretrained=True):
        super().__init__()

        self.backbone = timm.create_model('densenet121', pretrained=True, num_classes=2)

        in_features = self.backbone.classifier.in_features

        self.backbone.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x):
        return self.backbone(x)

In [ ]:
model = DenseNetClassifier()

In [ ]:
# Unfreeze classifier head
for param in model.backbone.classifier.parameters():
    param.requires_grad = True

# Unfreeze DenseNet denseblock4 as well
for name, param in model.backbone.named_parameters():
    if "denseblock4" in name:
        param.requires_grad = True

x = model(torch.randn(1, 3, 380, 380))
print(x.shape)

In [ ]:
from torch.amp import autocast, GradScaler


# criterion = FocalLoss(gamma=2.0, alpha=0.25, to_onehot_y=True, use_softmax=True, reduction="mean")
scaler = GradScaler(device="cuda")


def calculate_accuracy(logits, labels):
    preds = torch.argmax(logits, dim=1)
    return (preds == labels).float().mean().item()

def train_model(model, name="densenet_classifier"):
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    best_val_loss = float('inf')
    early_stop_patience = 5
    epochs_no_improve = 0


    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
    
    criterion = nn.CrossEntropyLoss(weight=torch.tensor([1.0, 2.0]).to(DEVICE), label_smoothing=0.1)
    
    print(f"Starting training for {name}")
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
    
        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Training - {name}]", leave=False):
            images = images.to(DEVICE)
            labels = labels.long().to(DEVICE)
    
            optimizer.zero_grad()
    
            with autocast(device_type='cuda'):
                outputs = model(images)                 # [B, 2]
                loss = criterion(outputs, labels)
    
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
    
            running_loss += loss.item()
    
        train_loss = running_loss / len(train_loader)
    
        model.eval()
        val_loss = 0.0
        val_acc = 0.0
    
        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validating - {name}]", leave=False):
                images = images.to(DEVICE)
                labels = labels.long().to(DEVICE)
    
                with autocast(device_type='cuda'):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
    
                val_loss += loss.item()
                val_acc += calculate_accuracy(outputs, labels)
    
        val_loss /= len(val_loader)
        val_acc /= len(val_loader)
    
        print(f"Epoch [{epoch+1}/{EPOCHS}] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
    
        scheduler.step(val_loss)
    
        if val_loss < best_val_loss:
            

            best_val_loss = val_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), f'{name}_best.pth')
            print(f" >>> Best model saved! (Val Loss: {val_loss:.4f})")
        else:
            epochs_no_improve += 1
            print(f"   --- No improvement for {epochs_no_improve}/{early_stop_patience} epochs.")
            if epochs_no_improve >= early_stop_patience:
                print(f"\n EARLY STOPPING at Epoch {epoch+1}. Best Val Loss: {best_val_loss:.4f}")
                break

    return history
    
    print(f"Training Complete for {name}")

In [ ]:
model.to(DEVICE)
history = train_model(model=model)

In [ ]:
def plot_train_vs_val(history, model_name="Model"):
    plt.figure(figsize=(7, 5))

    plt.plot(history["train_loss"], label="Train Loss")
    plt.plot(history["val_loss"], label="Val Loss")

    plt.title(f"{model_name}: Train vs Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

plot_train_vs_val(history, "DenseNet")

In [ ]:
print("Starting validation...")

results = {}
base_path = "/kaggle/working/"
name = "densenet_classifier"

print("\n==============================")
print(f" Evaluating: {name}")
print("==============================")

try:
    # Load best checkpoint
    ckpt_path = os.path.join(base_path, f"{name}_best.pth")

    if os.path.exists(ckpt_path):
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
        print(f"Loaded weights for {name}")
    else:
        print("Checkpoint not found. Using current model weights.")

    model = model.to(DEVICE)
    model.eval()

    all_labels = []
    all_preds = []
    all_probs = []

    # Validation loop
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Evaluating {name}"):
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            with autocast(device_type="cuda"):
                logits = model(images)
                probs = torch.softmax(logits, dim=1)

            preds = torch.argmax(probs, dim=1)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())

    # Convert to numpy
    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)

    # Metrics
    f1_macro = f1_score(all_labels, all_preds, average="macro", zero_division=0)

    try:
        auc_score = roc_auc_score(all_labels, all_probs)
    except:
        auc_score = float("nan")

    cm = confusion_matrix(all_labels, all_preds)

    print("-----------------------------------------------------")
    print(f" {name} RESULTS")
    print("-----------------------------------------------------")
    print(f"Macro F1-Score : {f1_macro:.4f}")
    print(f"AUC-ROC        : {auc_score:.4f}")
    print(classification_report(
        all_labels,
        all_preds,
        target_names=CLASS_NAMES,
        zero_division=0
    ))

    # Plot results
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=CLASS_NAMES
    ).plot(ax=axes[0], cmap="Blues", colorbar=False)

    axes[0].set_title(f"{name} - Confusion Matrix")

    if not np.isnan(auc_score):
        fpr, tpr, _ = roc_curve(all_labels, all_probs)
        axes[1].plot(fpr, tpr, label=f"AUC = {auc_score:.3f}")
        axes[1].plot([0, 1], [0, 1], "--")
        axes[1].set_title(f"{name} - ROC Curve")
        axes[1].set_ylabel("True Positive Rate")
        axes[1].legend()
        axes[1].grid(True)
    else:
        axes[1].text(0.5, 0.5, "ROC not available",
                     ha="center", va="center")
        axes[1].set_title("ROC Curve")

    plt.tight_layout()
    plt.show()

    # Save results
    results[name] = {
        "f1": f1_macro,
        "auc": auc_score
    }

except Exception as e:
    print(f"{name} evaluation failed: {e}")

In [ ]:
model = DenseNetClassifier()
model.load_state_dict(torch.load("/kaggle/working/densenet_classifier_best.pth", map_location=DEVICE))

In [ ]:
model = model.to(DEVICE)
model.eval()

sample_img = "/kaggle/working/output_folder/val/Tuberculosis/Tuberculosis-104.png"
img = Image.open(sample_img).convert('RGB')

input_tensor = val_test_transform(img).unsqueeze(0).to(DEVICE)

img_np = np.array(img.resize((224, 224))) / 255


with torch.no_grad():
    output = model(input_tensor)
    probs = torch.softmax(output, dim=1)
    pred_class = torch.argmax(probs, dim=1).item()
    confidence = probs[0][pred_class].item()

print(f"Prediction: {CLASS_NAMES[pred_class]}")
print(f"Confidence: {confidence:.4f}")

In [ ]:
# integrated gradients 

ig = IntegratedGradients(model)

attributions_ig, delta = ig.attribute(
    input_tensor,
    target=pred_class,
    n_steps=20,
    return_convergence_delta=True
)

# convert tensor to numpy
attr_ig = attributions_ig.squeeze().detach().cpu().numpy()

attr_ig = np.transpose(attr_ig, (1, 2, 0))

# average RGB channels
attr_ig = np.mean(np.abs(attr_ig), axis=2)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].imshow(img_np)
axes[0].set_title("Original Image")
axes[0].axis('off')

axes[1].imshow(attr_ig, cmap='hot')
axes[1].set_title("Integrated Gradients")
axes[1].axis("off")
plt.show()

In [ ]:
target_layer = model.backbone.features.denseblock4

# grad cam

guided_gc = GuidedGradCam(model, target_layer)
attribution_gc = guided_gc.attribute(input_tensor, target=pred_class)

attr_gc = attribution_gc.squeeze().detach().cpu().numpy()
attr_gc = np.transpose(attr_gc, (1, 2, 0))
attr_gc = np.mean(np.abs(attr_gc), axis=2)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].imshow(img_np)
axes[0].set_title("Original Image")
axes[0].axis('off')

axes[1].imshow(attr_gc, cmap='hot')
axes[1].set_title("Guided GradCAM")
axes[1].axis("off")
plt.show()

In [ ]:
model_cpu = model.cpu()
input_cpu = input_tensor.cpu()

occlusion = Occlusion(model_cpu)

attributions_occ = occlusion.attribute(
    input_cpu,
    target=pred_class,

    # movement step
    strides=(3, 16, 16),

    # occlusion patch size
    sliding_window_shapes=(3, 32, 32),

    baselines=0
)

attr_occ = attributions_occ.squeeze().detach().numpy()

# CHW → HWC
attr_occ = np.transpose(attr_occ, (1, 2, 0))

# average RGB channels
attr_occ = np.mean(np.abs(attr_occ), axis=2)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].imshow(img_np)
axes[0].set_title(f"Original Image\nPrediction: {CLASS_NAMES[pred_class]}")
axes[0].axis("off")
axes[1].imshow(attr_occ, cmap='hot')
axes[1].set_title("Occlusion Sensitivity")
axes[1].axis("off")
plt.tight_layout()
plt.show()